# Remesh script

In [3]:
import argparse
import trimesh
from stl import mesh

def repair_stl(input_file, output_file):
    try:
        # Load STL file using trimesh
        print(f"Loading {input_file}...")
        mesh_data = trimesh.load_mesh(input_file)

        # Check for issues and attempt repair
        if not mesh_data.is_watertight:
            print("Mesh is not watertight. Attempting repair...")
            mesh_data = mesh_data.fill_holes()

        if not mesh_data.is_winding_consistent:
            print("Mesh has inconsistent winding. Fixing...")
            mesh_data.fix_normals()

        # Remove duplicate and degenerate faces
        mesh_data.remove_duplicate_faces()
        mesh_data.remove_degenerate_faces()
        
        # Export repaired mesh
        mesh_data.export(output_file)
        print(f"Repaired mesh saved to {output_file}")

    except Exception as e:
        print(f"Error processing STL: {e}")

if __name__ == "__main__":
    # parser = argparse.ArgumentParser(description="Fix topology issues in an STL file.")
    # parser.add_argument("input", help="Path to the input STL file")
    # parser.add_argument("output", help="Path to save the fixed STL file")
    # args = parser.parse_args()

    Input = 'RaketTest3.stl'
    Output = 'RaketTest3_fixed.stl'
    repair_stl(Input, Output)

    # repair_stl(args.input, args.output)


Loading RaketTest3.stl...
Repaired mesh saved to RaketTest3_fixed.stl


C:\Users\Joël\AppData\Local\Temp\ipykernel_11540\1767035546.py:21: DeprecationWarning: `remove_duplicate_faces` is deprecated and will be removed in March 2024: replace with `mesh.update_faces(mesh.unique_faces())`
  mesh_data.remove_duplicate_faces()
C:\Users\Joël\AppData\Local\Temp\ipykernel_11540\1767035546.py:22: DeprecationWarning: `remove_degenerate_faces` is deprecated and will be removed in March 2024 replace with `self.update_faces(self.nondegenerate_faces(height=height))`
  mesh_data.remove_degenerate_faces()


In [6]:
import argparse
import trimesh
import pymeshfix
import numpy as np
import meshio

def remesh_stl(input_file, output_file, target_edge_length=2.0):
    try:
        print(f"Loading {input_file}...")
        mesh_data = trimesh.load_mesh(input_file)

        # Fix non-manifold geometry
        print("Fixing mesh topology...")
        mesh_fix = pymeshfix.MeshFix(mesh_data.vertices, mesh_data.faces)
        mesh_fix.repair(verbose=True)
        fixed_vertices, fixed_faces = mesh_fix.v, mesh_fix.f

        # Convert to trimesh format after repair
        fixed_mesh = trimesh.Trimesh(vertices=fixed_vertices, faces=fixed_faces)

        # Perform remeshing (subdividing long edges)
        print("Remeshing to improve uniformity...")
        fixed_mesh = fixed_mesh.subdivide_to_size(max_edge=target_edge_length)

        # Save the repaired and remeshed STL
        print(f"Saving repaired STL to {output_file}...")
        fixed_mesh.export(output_file)

        print("Remeshing completed successfully!")

    except Exception as e:
        print(f"Error processing STL: {e}")

if __name__ == "__main__":
    Input = 'RaketTest3.stl'
    Output = 'RaketTest3_remeshed.stl'
    remesh_stl(Input, Output)

    # remesh_stl(args.input, args.output, args.edge_length)


Loading RaketTest3.stl...
Fixing mesh topology...
Fixing degeneracies and intersections
Remeshing to improve uniformity...
Saving repaired STL to RaketTest3_remeshed.stl...
Remeshing completed successfully!
